In [ ]:
import pandas as pd
import scienceplots
import matplotlib.pyplot as plt

import logging
logging.basicConfig(level=logging.ERROR)
import os
import sys
import json
from collections import Counter
from rich import print
import numpy as np
plt.style.use(['science', 'no-latex', 'nature', 'grid'])

font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 12}

plt.rc('font', **font)

In [ ]:
def load_stats(save_dir):
    blueprints = []
    
    # list dir sorted by last modified 
    files = os.listdir(save_dir)
    files = sorted(files, key=lambda x: os.path.getmtime(f"{save_dir}/{x}"), reverse=False)
    #print(files)
    
    stats = {
        "succesfull": 0,
        "syntactic_passed": 0,
        "translatable_passed": 0,
        "simulation_passed": 0,
        "failed": 0,
        "iterations": [],
        "last_errors": [],
        "classification": [],
        "num_not_executeable": [],
        "iter_to_judge": [],
        "iter_to_sim": [],
    }
    
    for file in files[:50]:
        # load all json files
        try:
            if file.endswith(".json"):
                with open(f"{save_dir}/{file}") as f:
                    data = json.load(f)
                max_key = max([eval(key) for key in data["error_log"].keys() if key.isnumeric()])
                
                stats["succesfull"] += 1 if data["error_log"].get("succesfull", 0) else 0
                stats["syntactic_passed"] += 1 if data["error_log"].get("syntactic_passed", 0) else 0
                stats["translatable_passed"] += 1 if data["error_log"].get("translatable_passed", 0) else 0
                stats["simulation_passed"] += 1 if data["error_log"].get("simulation_passed", 0) else 0
                stats["failed"] += 1 if not data["error_log"]["succesfull"] else 0
                stats["classification"].append(data["classification"])
                if data["classification"] == "B":
                    blueprints.append(file)
                

                for key in data["error_log"].keys():
                    if key.isnumeric():
                        if data["error_log"][key].get("judge", False):
                            stats["num_not_executeable"].append(len(data["error_log"][key].get("judge", {}).get("missing_steps", {}).get("not_executable", [])))
                        
                        if not data["error_log"][key].get("Feedback", False):
                            stats["iter_to_sim"].append(int(key))
                            
                        elif data["error_log"][key].get("judge", False):
                            stats["iter_to_judge"].append(int(key) + 1)
                
                
                
                
                if data["error_log"]["succesfull"]:
                    stats["iterations"].append(max_key)
                    stats["last_errors"].append(0)
                else:
                    stats["iterations"].append(max_key)
                    stats["last_errors"].append(data["error_log"][str(max_key)]["Errors"])
        except Exception as e:
            continue
    return stats, blueprints

not_executeable_path = '../data/xdl_suggestions/xdl_features.csv'
not_executeable = pd.read_csv(not_executeable_path)
not_executeable.shape
        

##### Synonyms Pubchem

In [ ]:
import pubchempy as pcp
name = "Triethylenediamine"
cid = pcp.get_compounds(name, 'name')[0]
synonyms = pcp.get_synonyms(cid.cid)
len(synonyms[0]["Synonym"])

##### Figure 3 Knowledge Graph

Procedure Classification

In [ ]:
data_path = f"../data/memory/benchmark_10_papers_run_1/papers/"
papers_path = os.listdir(data_path)

# number of procedures per paper
num_procedures = {}


for paper in papers_path:
    with open(data_path + paper + "/ps_response.json", "r") as f:
        data = json.load(f)
        num_procedures[paper] = len(set(data["procedure_texts"]))
        

paper_path = "../data/memory/benchmark_10_papers_run_1/labbook/"
paper_labbook = os.listdir(paper_path)

paper_classification = {"A": 0, "B": 0, "C": 0}
for labbook in paper_labbook:
    with open(paper_path + labbook, "r") as f:
        data = json.load(f)
        paper_classification[data["classification"]] += 1
        
# number of procedures from the german praktikum

data_path = f"../data/memory/benchmark_german_praktikum/papers/"
papers_path = os.listdir(data_path)

# number of procedures per paper
num_procedures_german = {}

for paper in papers_path:
    with open(data_path + paper + "/ps_response.json", "r") as f:
        data = json.load(f)
        num_procedures_german[paper] = len(set(data["procedure_texts"]))
        
print(num_procedures_german)

practical_labbook_path = "../data/memory/benchmark_german_praktikum/labbook/"
practical_labbook = os.listdir(practical_labbook_path)

practical_classification = {"A": 0, "B": 0, "C": 0}
for labbook in practical_labbook:
    with open(practical_labbook_path + labbook, "r") as f:
        data = json.load(f)
        practical_classification[data["classification"]] += 1
        
        
print(practical_classification)

data_path = f"../data/memory/benchmark_thesis_run_1/papers/"
papers_path = os.listdir(data_path)

# number of procedures per paper
num_procedures_thesis = {}
 
for paper in papers_path:
    with open(data_path + paper + "/ps_response.json", "r") as f:
        data = json.load(f)
        num_procedures_thesis[paper] = len(set(data["procedure_texts"]))
        
print(num_procedures_thesis)


thesis_labbook_path = "../data/memory/benchmark_thesis_run_1/labbook/"
thesis_labbook = os.listdir(thesis_labbook_path)

thesis_classification = {"A": 0, "B": 0, "C": 0}
for labbook in thesis_labbook:
    with open(thesis_labbook_path + labbook, "r") as f:
        data = json.load(f)
        thesis_classification[data["classification"]] += 1
        
print(thesis_classification)

In [ ]:
# plot classifications of thesis and german praktikum

fig, ax = plt.subplots(figsize=(6, 2))

# grouped bar plot
barWidth = 0.25

bars1 = [paper_classification["A"], practical_classification["A"], thesis_classification["A"]]
bars2 = [paper_classification["B"], practical_classification["B"], thesis_classification["B"]]
bars3 = [paper_classification["C"], practical_classification["C"], thesis_classification["C"]]

r1 = range(len(bars1))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]

ax.bar(r1, bars1, color='g', width=barWidth, edgecolor='grey', label='executable')
ax.bar(r2, bars2, color='y', width=barWidth, edgecolor='grey', label='blueprint')
ax.bar(r3, bars3, color='black', width=barWidth, edgecolor='grey', label='incomplete')

# numbers on top of the bars
for i, v in enumerate(zip(bars1, bars2, bars3)):
    ax.text(i, v[0] + 5, str(v[0]), color='black', ha='center')
    ax.text(i + barWidth, v[1] + 5, str(v[1]), color='black', ha='center')
    ax.text(i + 2 * barWidth, v[2] + 5, str(v[2]), color='black', ha='center')
    



# Add xticks on the middle of the group bars
ax.set_ylabel('number of procedures', fontweight='bold', fontsize=10)
ax.set_xticks([r + barWidth for r in range(len(bars1))])
ax.set_xticklabels(["Publications", "German\npractical", "PhD\nthesis"])


plt.ylim(0, 510)
plt.yticks([0, 100, 200, 300, 400] ,fontsize=10)
plt.xticks(fontsize=10)

# remove minor ticks
plt.minorticks_off()
plt.grid(axis='x')

plt.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))


ax.bar(num_procedures.keys(), num_procedures.values(), color='g', edgecolor='grey')

# annotate with black line and paper number on top of the bar
for i, v in enumerate(num_procedures.values()):
    ax.text(i, v + 5, str(v), color='black', ha='center')
    
ax.text(4.5, 220, "Publications", color='black', ha='center', fontsize=10)
ax.hlines(210, 0, 9, color='black', linestyles='dashed', linewidth=1)

# from german praktikum
number_papers = len(num_procedures.keys())
ax.bar([str(number_papers)], list(num_procedures_german.values())[-1], color='y', edgecolor='grey')

# annotate with black line and paper number on top of the bar
for i, v in enumerate(list(num_procedures_german.values())):
    ax.text(i + number_papers, v + 5, str(v), color='black', ha='center')
    
    # ax.text(i + number_papers, v + 50, "German\npractical", color='black', ha='center', rotation=90, fontsize=10)
    

# from phd thesis
number_papers_and_prak = len(num_procedures.keys()) + 1
ax.bar([str(number_papers_and_prak)], list(num_procedures_thesis.values()), color='k', edgecolor='grey')

# annotate with black line and paper number on top of the bar
for i, v in enumerate(list(num_procedures_thesis.values())):
    ax.text(i + number_papers_and_prak, v + 5, str(v), color='black', ha='center')
    # ax.text(i + number_papers_and_prak, v + 50, "PhD\nthesis", color='black', ha='center', rotation=90, fontsize=10)


ax.set_ylabel("number of procedures", fontsize=10, fontweight='bold')
ax.set_xlabel(r"paper no.", fontsize=10, fontweight='bold')


 
plt.yticks([0, 50, 100, 150, 200],fontsize=10)
plt.xticks(fontsize=10)

plt.minorticks_off()
plt.grid(axis='x')


ax.set_ylim(0, 300)
plt.legend(["Publications", "German practical", "PhD thesis"], loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3 , fontsize=10)

plt.show()

Analytical Data Extraction

In [ ]:
colors = plt.cm.tab20.colors

In [ ]:
results = {
    "F1": 0.85213582,
    "Precision": 0.896313364,
    "Recall": 0.812108559
}

fig4, ax4 = plt.subplots(figsize=(4, 3))

results_df = pd.DataFrame(results, index=[0]).T
results_df.plot(kind="bar", ax=ax4, color=colors, capsize=4, stacked=True)

# put values rounded to 3 decimal places on top of bars
for p in ax4.patches:
    ax4.annotate(
        f"{p.get_height():.3f}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=14,
    )

plt.grid()
plt.xticks(rotation=0, ha="center")
plt.ylim(0, 1.1)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.minorticks_off()

# turn off legend
plt.legend().set_visible(False)

plt.tight_layout()

plt.tight_layout()
plt.show()

Chemical Extraction (OrgSyn Papers)

In [ ]:
# Categories and positions
metrics = ["Precision", "Recall", "F1"]
x = np.arange(len(metrics))
width = 0.25

# Data
paper_mean = [0.986916281, 0.96467863, 0.975228148]
paper_std = [0.033071891, 0.060092521, 0.041395467]

procedure_mean = [1.0, 0.930604859, 0.962769733]
procedure_std = [0.0, 0.09145335, 0.051465699]

cde_mean = [0.90261088, 0.883754314, 0.890171849]
cde_std = [0.091926789, 0.081009259, 0.05172468]

# Styling to match example
plt.figure(figsize=(4, 3))

error_kw = dict(elinewidth=1.5, capsize=4, capthick=1.5)

plt.bar(
    x - width,
    paper_mean,
    width,
    yerr=paper_std,
    label="From Paper",
    color="#1f77b4",
    error_kw=error_kw,
)
plt.bar(
    x,
    procedure_mean,
    width,
    yerr=procedure_std,
    label="From Procedure",
    color="#aec7e8",
    error_kw=error_kw,
)
plt.bar(
    x + width,
    cde_mean,
    width,
    yerr=cde_std,
    label="CDE2.0",
    color="#2ca02c",
    error_kw=error_kw,
)

plt.xticks(x, metrics, fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0, 1.05)
plt.legend(fontsize=11, frameon=True)
plt.tight_layout()
plt.savefig("metrics_comparison.svg", format="svg", bbox_inches="tight")
plt.show()


##### Figure 4C translation success

In [ ]:
stats = load_stats("../data/memory/benchmark_50_ChemRND/labbook/")[0]
stats_2 = load_stats("../data/memory/benchmark_50_ChemRND_run_2/labbook/")[0]
stats_3 = load_stats("../data/memory/benchmark_50_ChemRND_run_3/labbook/")[0]

In [ ]:
# calculate standard deviation given n values

def std_deviation(values):
    n = len(values)
    mean = sum(values) / n
    return (sum((x - mean) ** 2 for x in values) / n) ** 0.5

In [ ]:
num_procedures = 50

In [ ]:
all_iter_keys = sorted(list(set((list(Counter(stats_3["iterations"]).keys())) + list(Counter(stats["iterations"]).keys()) + list(Counter(stats_2["iterations"]).keys()))))
all_iter_to_judge_keys = sorted(list(set(list(Counter(stats_3["iter_to_judge"]).keys()) + list(Counter(stats["iter_to_judge"]).keys()) + list(Counter(stats_2["iter_to_judge"]).keys()))))
all_iter_to_sim_keys = sorted(list(set(list(Counter(stats_3["iter_to_sim"]).keys()) + list(Counter(stats["iter_to_sim"]).keys()) + list(Counter(stats_2["iter_to_sim"]).keys()))))
all_num_not_executeable_keys = sorted(list(set(list(Counter(stats_3["num_not_executeable"]).keys()) + list(Counter(stats["num_not_executeable"]).keys()) + list(Counter(stats_2["num_not_executeable"]).keys()))))

data_stats = {
    "syntactic_passed": sum([stats["syntactic_passed"], stats_2["syntactic_passed"], stats_3["syntactic_passed"]]) / 3,
    "syntactic_passed_std": std_deviation([stats["syntactic_passed"], stats_2["syntactic_passed"], stats_3["syntactic_passed"]]),
    "simulation_passed": sum([stats["simulation_passed"], stats_2["simulation_passed"], stats_3["simulation_passed"]]) / 3,
    "simulation_passed_std": std_deviation([stats["simulation_passed"], stats_2["simulation_passed"], stats_3["simulation_passed"]]),
    "syntactic_failed": sum([num_procedures - stats["syntactic_passed"], num_procedures - stats_2["syntactic_passed"], num_procedures - stats_3["syntactic_passed"]]) / 3,
    "syntactic_failed_std": std_deviation([num_procedures - stats["syntactic_passed"], num_procedures - stats_2["syntactic_passed"], num_procedures - stats_3["syntactic_passed"]]),
    "simulation_failed": sum([stats["failed"] - (num_procedures - stats["syntactic_passed"]), stats_2["failed"] - (num_procedures - stats_2["syntactic_passed"]), stats_3["failed"] - (num_procedures - stats_3["syntactic_passed"])]) / 3,
    "simulation_failed_std": std_deviation([stats["failed"] - (num_procedures - stats["syntactic_passed"]), stats_2["failed"] - (num_procedures - stats_2["syntactic_passed"]), stats_3["failed"] - (num_procedures - stats_3["syntactic_passed"])]),
    # calculate the mean and standard deviation of how many iterations it took to complete the procedure for each number of iterations
    "iterations_std": [std_deviation([Counter(stats["iterations"])[key], Counter(stats_2["iterations"])[key], Counter(stats_3["iterations"])[key],]) for key in all_iter_keys],
    "iterations_mean": [sum([Counter(stats["iterations"])[key], Counter(stats_2["iterations"])[key], Counter(stats_3["iterations"])[key],]) / 3 for key in all_iter_keys],
    "iter_to_judge_mean": [sum([Counter(stats["iter_to_judge"])[key], Counter(stats_2["iter_to_judge"])[key], Counter(stats_3["iter_to_judge"])[key]]) / 3 for key in all_iter_to_judge_keys],
    "iter_to_judge_std": [std_deviation([Counter(stats["iter_to_judge"])[key], Counter(stats_2["iter_to_judge"])[key], Counter(stats_3["iter_to_judge"])[key]]) for key in all_iter_to_judge_keys],
    "iter_to_sim_mean": [sum([Counter(stats["iter_to_sim"])[key], Counter(stats_2["iter_to_sim"])[key], Counter(stats_3["iter_to_sim"])[key]]) / 3 for key in all_iter_to_sim_keys],
    "iter_to_sim_std": [std_deviation([Counter(stats["iter_to_sim"])[key], Counter(stats_2["iter_to_sim"])[key], Counter(stats_3["iter_to_sim"])[key]]) for key in all_iter_to_sim_keys],
    "num_not_executeable_mean": [sum([Counter(stats["num_not_executeable"])[key], Counter(stats_2["num_not_executeable"])[key], Counter(stats_3["num_not_executeable"])[key]]) / 3 for key in all_num_not_executeable_keys],
    "num_not_executeable_std": [std_deviation([Counter(stats["num_not_executeable"])[key], Counter(stats_2["num_not_executeable"])[key], Counter(stats_3["num_not_executeable"])[key]]) for key in all_num_not_executeable_keys],
}


In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# import seaborn as sns
# sns.set_theme(style="whitegrid")
# # paper style
# sns.set_context("paper")
# bar width
barWidth = 0.5

plt.bar(
    [
        "total",
        r"$X$DL valid",
        r"$X$DL invalid",
        "simulation passed",
        "simulation failed",
    ],
    [
        num_procedures * 2,
        data_stats["syntactic_passed"] * 2,
        data_stats["syntactic_failed"] * 2,
        data_stats["simulation_passed"] * 2,
        data_stats["simulation_failed"] * 2,
    ],
    
    # add std
    yerr=[
        0,
        data_stats["syntactic_passed_std"] * 2,
        data_stats["syntactic_failed_std"] * 2,
        data_stats["simulation_passed_std"] * 2,
        data_stats["simulation_failed_std"] * 2,
    ],
    
    # change yerr style
    capsize=5,
    ecolor='black',
    color=["grey"],
    edgecolor='k',
    width=barWidth,
)

# give nice style
plt.ylabel("procedures [%]", fontsize=10, fontweight='bold')
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)

# hide minor ticks
plt.minorticks_off()

plt.ylim(0)
# remove vertical grid
plt.grid(axis='x')


# to svg
plt.savefig("synthesis_procedures.svg", format="svg")

##### Figure 4D Translation Benchmark

In [ ]:
data = "../data/benchmark_translation.xlsx"
df = pd.read_excel(data, sheet_name="Sheet1")
df.dropna(inplace=True)


source_mapping = {
    "paper": "ACRA$_{paper}$",
    "procedure_wj": "ACRA$_{procedure}$",
    "procedure_nj": "ACRA$_{procedure-no-judge}$",
    "synthreader": "Synthreader",
}

df["source"] = df["source"].map(source_mapping)

# group by source, then metric
grouped = df.groupby(["metric", "source"])

In [ ]:

fig, ax = plt.subplots(figsize=(13, 4))

colors = plt.cm.tab20.colors

# Define colors for each source
# colors = [source_to_color[source] for source in df["source"].unique()]
# Clip yerr values to a maximum of 1
mean = grouped.mean()
std = grouped.std()
yerr = std.unstack().clip(upper=1)

# Ensure mean + yerr does not exceed 1
yerr = yerr.where(mean.unstack() + yerr <= 1, 1 - mean.unstack())

# Plotting
print()
mean.unstack().plot(
    kind="bar",
    yerr=yerr,
    ax=ax,
    color=colors,
    capsize=4,
    error_kw=dict(elinewidth=2, ecolor="black"),
)

# Rotate x-axis labels for better readability
plt.xticks(rotation=0, ha="center")

# Add legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))

plt.ylim(0, 1.1)
ax.legend(
    by_label.values(),
    [v.split(",")[-1].replace(")", "").strip() for v in by_label.keys()],
    bbox_to_anchor=(0.5, 1.05),
    loc="center",
    fontsize=16,
    ncol=4  # Stretch the legend by increasing the number of columns
)


plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel(None)
plt.grid()
plt.minorticks_off()

plt.tight_layout()

plt.show()


##### Figure 5

In [ ]:
xdl_memory = [
    load_stats(f"../data/memory/benchmark_25_procedures_with_memory_{i}/labbook") for i in range(1, 4)
]


no_xdl_memory = [
    load_stats(f"../data/memory/benchmark_25_procedures_without_memory_{i}/labbook") for i in range(1, 4)
]

synthetic_memory = [
    load_stats(f"../data/memory/benchmark_25_procedures_without_memory_with_expansion_{i}/labbook") for i in range(1, 4)
]

no_external_data = [
    load_stats(f"../data/memory/benchmark_25_procedures_no_external_data_{i}/labbook") for i in range(1, 4)
]

In [ ]:
no_external_data = [
    load_stats(f"../data/memory/benchmark_25_procedures_no_external_data_{i}/labbook") for i in range(1, 4)
]

###### Translations external data abliation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.5))

# grouped bar plot
barWidth = 0.15

bars_success = [25, 25, 25 ,25]

bars_syntactic = [sum([
    xdl_memory[0][0]["syntactic_passed"], 
    xdl_memory[1][0]["syntactic_passed"], 
    xdl_memory[2][0]["syntactic_passed"]
]) /3, sum([
    synthetic_memory[0][0]["syntactic_passed"], 
    synthetic_memory[1][0]["syntactic_passed"], 
    synthetic_memory[2][0]["syntactic_passed"]
]) /3, sum([
    no_xdl_memory[0][0]["syntactic_passed"], 
    no_xdl_memory[1][0]["syntactic_passed"], 
    no_xdl_memory[2][0]["syntactic_passed"]
]) /3, sum([
    no_external_data[0][0]["syntactic_passed"], 
    no_external_data[1][0]["syntactic_passed"], 
    no_external_data[2][0]["syntactic_passed"]
]) /3]



bars_simulation = [sum([
    xdl_memory[0][0]["simulation_passed"], 
    xdl_memory[1][0]["simulation_passed"], 
    xdl_memory[2][0]["simulation_passed"]
]) /3, sum([
    synthetic_memory[0][0]["simulation_passed"], 
    synthetic_memory[1][0]["simulation_passed"], 
    synthetic_memory[2][0]["simulation_passed"]
]) /3, sum([
    no_xdl_memory[0][0]["simulation_passed"], 
    no_xdl_memory[1][0]["simulation_passed"], 
    no_xdl_memory[2][0]["simulation_passed"]
]) /3, sum([
    no_external_data[0][0]["simulation_passed"], 
    no_external_data[1][0]["simulation_passed"], 
    no_external_data[2][0]["simulation_passed"]
]) /3]

bars_failed = [sum([
    xdl_memory[0][0]["failed"], 
    xdl_memory[1][0]["failed"], 
    xdl_memory[2][0]["failed"]
]) /3, sum([
    synthetic_memory[0][0]["failed"], 
    synthetic_memory[1][0]["failed"], 
    synthetic_memory[2][0]["failed"]
]) /3, sum([
    no_xdl_memory[0][0]["failed"], 
    no_xdl_memory[1][0]["failed"], 
    no_xdl_memory[2][0]["failed"]
]) /3, sum([
    no_external_data[0][0]["failed"], 
    no_external_data[1][0]["failed"], 
    no_external_data[2][0]["failed"]
]) /3]


# Set position of bar on X axis
r1 = range(len(bars_success))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]
r4 = [x + barWidth for x in r3]
# r5 = [x + barWidth for x in r4]

# scale to 100
bars_success = np.array(bars_success) * 4
bars_syntactic = np.array(bars_syntactic) * 4
# bars_translatable = np.array(bars_translatable) * 4
bars_simulation = np.array(bars_simulation) * 4
bars_failed = np.array(bars_failed) * 4


# Make the plot
plt.bar(r1, bars_success, color='g', width=barWidth, edgecolor='grey', label='Total')
plt.bar(r2, bars_syntactic, color='b', width=barWidth, edgecolor='grey', label=r'$X$DL valid')
plt.bar(r3, bars_simulation, color='y', width=barWidth, edgecolor='grey', label='Successful')
# plt.bar(r3, bars_translatable, color='r', width=barWidth, edgecolor='grey', label='Translatable')
plt.bar(r4, bars_failed, color='k', width=barWidth, edgecolor='grey', label='Failed')

# number over each bar
for i in range(4):
    plt.text(i, bars_success[i] + 0.5, round(bars_success[i], 1), ha='center', va='bottom')
    plt.text(i + barWidth, bars_syntactic[i] + 0.5, round(bars_syntactic[i], 1), ha='center', va='bottom')
    # plt.text(i + 2*barWidth, bars_translatable[i] + 0.5, round(bars_translatable[i], 1), ha='center', va='bottom')
    plt.text(i + 2*barWidth, bars_simulation[i] + 0.5, round(bars_simulation[i], 1), ha='center', va='bottom')
    plt.text(i + 3*barWidth, bars_failed[i] + 0.5, round(bars_failed[i], 1), ha='center', va='bottom')

# Add xticks on the middle of the group bars
plt.xticks([r + barWidth for r in range(len(bars_success))], [r'$X$DL memory', r'No initial $X$DL memory', r'No $X$DL memory','No external data'], fontsize=10)
# only y ticks to 100
plt.yticks([0, 20, 40, 60, 80, 100],fontsize=10)
plt.ylabel('procedures [%]', fontsize=10, fontweight='bold')

# remove minor ticks
plt.minorticks_off()
plt.grid(axis='x')


plt.ylim(0, 150)

# Create legend & Show graphic horizontal
plt.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=4)

# save svg
plt.savefig("memory_success.svg", format="svg")

##### Figure 6 suggested steps

In [ ]:
import pandas as pd
import logging
logging.basicConfig(level=logging.ERROR)
import numpy as np

In [ ]:
import scienceplots
import matplotlib.pyplot as plt
plt.style.use(['science', 'no-latex', 'nature', 'grid'])

font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 10}

plt.rc('font', **font)

In [ ]:
# get current color map
cmap = plt.get_cmap()
colors = cmap(np.linspace(0, 1, 10))

In [ ]:
suggested_df = pd.read_csv('../data/suggested_steps/suggested_steps.csv', index_col=None)

#sort by frequency column
suggested_df = suggested_df.sort_values(by=['type','frequency'], ascending=False, ignore_index=True)
# reset index

suggested_df.head()

In [ ]:
# bar plot of suggested steps with frequency and colored by value of type column

fig, ax = plt.subplots(1, 1, figsize=(12, 2.5))
suggested_df['frequency'].plot(kind='bar', ax=ax, color=suggested_df['type'].map({1: colors[3], 2: colors[5], 3: colors[-1]}), edgecolor='black')
# step_types over bar plot
for i, row in suggested_df.iterrows():
    ax.text(i, row['frequency'] + 1, row['step_type'], rotation=90, ha='center', va='bottom', fontsize=8)
    
# log scale
# ax.set_yscale('log')
# remove x-axis ticks
ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
ax.set_yticks([0, 10, 20, 30, 40, 50])

#remove minor ticks
ax.minorticks_off()
#remove vertical gridlines
ax.grid(axis="both")

plt.ylim(0, 65)
# remove upper and right spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

# save as svg
plt.savefig('suggested_steps.svg', bbox_inches='tight')

##### SI Figures Classification

In [ ]:
classification = {
    "A (executable)": {"f1": 0.53846, "precision": 1, "recall": 0.6},
    "B (blueprint)": {"f1": 1, "precision": 0.18182, "recall": 0.30769},
    "C (incomplete)": {"f1": 1, "precision": 0.78571, "recall": 0.88},
}

missclassification = {
    "no synthesis information": 3 / 12,
    "reagent placeholder": 7 / 12,
    "reference to general procedure": 2 / 12,
}

correct_classification = {"correct": 27 /40, "incorrect": 13/40}


# plot classification
fig, ax = plt.subplots(figsize=(13, 4))

classification_df = pd.DataFrame(classification).T
missclassification_df = pd.DataFrame(missclassification, index=[0]).T
correct_classification_df = pd.DataFrame(correct_classification, index=[0]).T


classification_df.plot(kind="bar", ax=ax, color=colors, capsize=4)
# reset colors 
colors = plt.cm

plt.grid()
plt.xticks(rotation=0, ha="center")
plt.ylim(0, 1.1)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.xlabel(None)
plt.legend(
    ["F1", "Precision", "Recall"],
    bbox_to_anchor=(0.5, 1.05),
    loc="center",
    fontsize=14,
    ncol=3,
)
plt.minorticks_off()
plt.tight_layout()
plt.show()


# More muted colors for pie charts using pastel colors
pastel_colors = ['#95a5a6', '#bdc3c7', '#ecf0f1']

# missclassification
fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.pie(
    missclassification_df[0],
    labels=missclassification_df.index,
    autopct="%.1f%%",
    colors=pastel_colors,
    textprops={"fontsize": 12},
    radius=1.
)
ax2.set_aspect('equal')
ax2.set_title("Misclassification Reasons", fontsize=12)
plt.tight_layout()
plt.show()

fig3, ax3 = plt.subplots(figsize=(6, 4))
ax3.pie(
    correct_classification_df[0],
    labels=correct_classification_df.index,
    autopct="%.1f%%",
    colors=pastel_colors[:2],
    textprops={"fontsize": 12},
    radius=1.
)
ax3.set_aspect('equal')
ax3.set_title("Classification Accuracy", fontsize=12)
plt.tight_layout()
plt.show()


##### SI Figure translation efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# iterations distribution
plt.bar(
    range(1, 7),
    np.array(data_stats["iterations_mean"]) * 2,
    yerr=np.array(data_stats["iterations_std"]) * 2,
    capsize=5,
    ecolor="black",
    color="grey",
    edgecolor="k",
    width=barWidth,
)
plt.xlabel("number of iterations", fontsize=10, fontweight="bold")
plt.ylabel("procedures [%]", fontsize=10, fontweight="bold")
plt.xticks(range(1, 7), fontsize=10)
plt.yticks(fontsize=10)

# disable minor ticks
plt.minorticks_off()

# remove vertical grid
plt.grid(axis="x")


# to svg
plt.savefig("synthesis_iterations.svg", format="svg")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# iterations to judge distribution
plt.bar(
    range(1, 5),
    data_stats["iter_to_judge_mean"],
    yerr=data_stats["iter_to_judge_std"],
    capsize=5,
    ecolor="black",
)
plt.xlabel(
    "number of iterations to discrepancy check", fontsize=10, fontweight="bold"
)
plt.ylabel("number of procedures", fontsize=10, fontweight="bold")

plt.xticks(range(1, 5), fontsize=10)
plt.yticks(fontsize=10)

# to svg
plt.savefig("synthesis_iterations_to_judge.svg", format="svg")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# iterations to simulation distribution
plt.bar(range(1,7), data_stats["iter_to_sim_mean"], yerr=data_stats["iter_to_sim_std"], capsize=5, ecolor='black')
plt.xlabel("number of iterations to simulation", fontsize=10, fontweight='bold')
plt.ylabel("number of procedures", fontsize=10, fontweight='bold')

plt.xticks(range(1, 7), fontsize=10)
plt.yticks(fontsize=10)

# to svg
plt.savefig("synthesis_iterations_to_sim.svg", format="svg")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# not translatable procedures
# bar width is 0.8
plt.bar(
    ["total", "has not executeable step"],
    [
        num_procedures,
        sum(
            [
                num_procedures - stats["translatable_passed"],
                num_procedures - stats_2["translatable_passed"],
                num_procedures - stats_3["translatable_passed"],
            ]
        )
        / 3,
    ],
    yerr=std_deviation(
        [
            num_procedures - stats["translatable_passed"],
            num_procedures - stats_2["translatable_passed"],
            num_procedures - stats_3["translatable_passed"],
        ]
    ),
    capsize=5,
    ecolor="black",
)
plt.ylabel("number of procedures", fontsize=10, fontweight="bold")
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# to svg
plt.savefig("synthesis_translatable.svg", format="svg")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))
# set font to something science like


plt.rcParams["font.family"] = "times new roman"
plt.rcParams["font.weight"] = "normal"




# number of not executeable steps
plt.bar(range(0, 5), data_stats["num_not_executeable_mean"], yerr=data_stats["num_not_executeable_std"], capsize=5, ecolor='black')
plt.xlabel("number of not executeable steps", fontsize=10, fontweight="bold")
plt.ylabel("number of procedures", fontsize=10, fontweight="bold")
plt.xticks(range(0, 4))

# make text larger
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# limit y axis
plt.ylim(0)


# to svg
plt.savefig("synthesis_not_executeable.svg", format="svg")

##### SI classification JChem

In [ ]:
import logging
logging.basicConfig(level=logging.ERROR)
import os
import sys
from rich import print
import time
import json
import random
from rich.progress import track

sys.path.append("..")
from acra import paper_to_xdl, check_xdl
from acra.utils import run_xdl_simulation

import matplotlib.pyplot as plt 

In [ ]:
data_path = f"C:/code/acra_gl/chemaeleon/data/memory/benchmark_paper_jchem/papers/"
papers_path = os.listdir(data_path)

# number of procedures per paper
num_procedures = {}


for paper in papers_path:
    with open(data_path + paper + "/ps_response.json", "r") as f:
        data = json.load(f)
        num_procedures[paper] = len(set(data["procedure_texts"]))
        
        
paper_path = f"C:/code/acra_gl/chemaeleon/data/memory/benchmark_paper_jchem/labbook/"
paper_labbook = os.listdir(paper_path)

paper_classification = {"A": 0, "B": 0, "C": 0}
for labbook in paper_labbook:
    with open(paper_path + labbook, "r") as f:
        data = json.load(f)
        keys = list(data.keys())
        # print(keys)
        paper_classification[data[keys[1]]["classification"]] += 1
        
paper_classification

In [ ]:
import scienceplots
import matplotlib.pyplot as plt
plt.style.use(['science', 'no-latex', 'nature', 'grid'])

font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 10}

plt.rc('font', **font)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))


ax.bar(num_procedures.keys(), num_procedures.values(), color='g', edgecolor='grey')

# annotate with black line and paper number on top of the bar
for i, v in enumerate(num_procedures.values()):
    ax.text(i, v + 5, str(v), color='black', ha='center')
    
# ax.text(4.5, 220, "Publications", color='black', ha='center', fontsize=10)
# ax.hlines(210, 0, 9, color='black', linestyles='dashed', linewidth=1)

# from german praktikum
# number_papers = len(num_prc:\code\acra_gl\chemaeleon\notebooks\paper_benchmark.ipynbocedures.keys())
# ax.bar([str(number_papers)], list(num_procedures_german.values())[-1], color='y', edgecolor='grey')

# # annotate with black line and paper number on top of the bar
# for i, v in enumerate(list(num_procedures_german.values())):
#     ax.text(i + number_papers, v + 5, str(v), color='black', ha='center')
    
#     # ax.text(i + number_papers, v + 50, "German\npractical", color='black', ha='center', rotation=90, fontsize=10)
    

# # from phd thesis
# number_papers_and_prak = len(num_procedures.keys()) + 1
# ax.bar([str(number_papers_and_prak)], list(num_procedures_thesis.values()), color='k', edgecolor='grey')

# # annotate with black line and paper number on top of the bar
# for i, v in enumerate(list(num_procedures_thesis.values())):
#     ax.text(i + number_papers_and_prak, v + 5, str(v), color='black', ha='center')
#     # ax.text(i + number_papers_and_prak, v + 50, "PhD\nthesis", color='black', ha='center', rotation=90, fontsize=10)


ax.set_ylabel("number of procedures", fontsize=10, fontweight='bold')
ax.set_xlabel(r"paper no.", fontsize=10, fontweight='bold')


 
plt.yticks([0, 50, 100, 150, 200],fontsize=10)
plt.xticks(fontsize=10)

plt.minorticks_off()
plt.grid(axis='x')


ax.set_ylim(0, 300)
plt.legend(["Publications", "German practical", "PhD thesis"], loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3 , fontsize=10)

plt.show()